# Structure for project    




In [ ]:
import sys, importlib, types

# Create a fake imp module for compatibility
imp = types.ModuleType("imp")
imp.reload = importlib.reload
sys.modules["imp"] = imp

# Now load autoreload
%load_ext autoreload
%autoreload 2


import sys, os
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Dl_mostfa_saad/volleyball_project"
# remove HuggingFace datasets if already cached
if "datasets" in sys.modules:
    del sys.modules["datasets"]

# force project root to be first
sys.path.insert(0, PROJECT_ROOT)

print(sys.path[:3])


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
['/content/drive/MyDrive/Colab Notebooks/Dl_mostfa_saad/volleyball_project', '/content/drive/MyDrive/Colab Notebooks/Dl_mostfa_saad/volleyball_project', '/content']


In [ ]:
%pwd


'/content/drive/MyDrive/Colab Notebooks/Dl_mostfa_saad/volleyball_project'

In [ ]:

%cd '/content/drive/MyDrive/Colab Notebooks/Dl_mostfa_saad/volleyball_project'


/content/drive/MyDrive/Colab Notebooks/Dl_mostfa_saad/volleyball_project


In [ ]:
import yaml
import torch
from torch.utils.data import DataLoader
from torch import optim
import os

from dataset.raw_dataset import VolleyballRawDataset
from dataset.registry import  ADAPTER_REGISTRY

from models.backbones.resnet50 import ResNet50
from models.baseline_model1.baseline1 import Baseline1

from trainers.trainer import Trainer
from dataset.transforms import transforms_frame_train, transforms_frame_val

from utils.checkpoint import  save_checkpoint,load_checkpoint
from utils.visualization import  plot_metrics
from utils.save_prediction import  save_predictions
from utils.seed import  set_seed
from utils.metrics import accuracy




In [ ]:
transform_train = transforms_frame_train()
transform_val   = transforms_frame_val()
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = yaml.safe_load(open("configs/baseline1.yaml"))

root=cfg["dataset"]["root"]
split_train=cfg["splits"]["train"]
split_val=cfg["splits"]["val"]
split_test=cfg["splits"]["test"]

baseline=cfg["model"]["baseline"]

num_workers=os.cpu_count()
batch_size_train=cfg["dataloader"]["train_batch_size"]
batch_size_val=cfg["dataloader"]["val_batch_size"]
num_classes=cfg["model"]["num_classes"]
lr=cfg["optimizer"]["lr"]
weight_decay=cfg["optimizer"]["weight_decay"]


output_dir=cfg["output"]["root"]

epochs=cfg["trainer"]["epochs"]

In [ ]:
# Raw dataset
train_raw = VolleyballRawDataset(
    root,
    split_train,
)


In [ ]:

val_raw = VolleyballRawDataset(
     root,
    split_val,
)


In [ ]:

test_raw = VolleyballRawDataset(
     root,
    split_test,
)


In [ ]:
# Adapter selection
Adapter = ADAPTER_REGISTRY[baseline]


In [ ]:
train_ds = Adapter(train_raw, transform_train)
val_ds = Adapter(val_raw,transform_val )
test_ds=Adapter(test_raw,transform_val )

In [ ]:

train_loader = DataLoader(
    train_ds,
    batch_size_train,
    shuffle=True,
    num_workers=num_workers,
)


val_loader = DataLoader(
    val_ds,
    batch_size_val,
    shuffle=False,
    num_workers=num_workers
)
test_loader = DataLoader(
    test_ds,
    batch_size_val,
    shuffle=False,
    num_workers=num_workers
)


In [ ]:

# Model
model = Baseline1(
    backbone=ResNet50(),
    num_classes=num_classes
)
model = model.to(device)
optimizer = optim.AdamW(
filter(lambda p: p.requires_grad, model.parameters()),
lr=lr,
weight_decay=weight_decay)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer = optimizer, T_max=epochs,
    eta_min=1e-6)



Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 149MB/s]


In [ ]:

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=torch.nn.CrossEntropyLoss(),
    accuracy=accuracy,
    save_checkpoint=save_checkpoint,
    device=device,
    output_dir=output_dir,
    epochs=epochs
)

trainer.train()

KeyboardInterrupt: 

In [ ]:
trainer.plot_metrics()


In [ ]:
trainer.save_predictions(model,test_loader,device,output_dir)


In [ ]:
trainer.load_checkpoint(model,optimizer,f"{output_dir}/best_model.pth",device)
